# 03 - Fase 3: mejoras de la GAN

Este notebook reúne las mejoras de la GAN base de la Fase 3. Cada mejora se entrena con **el mismo protocolo** que las variantes del notebook 02 (datos de **Antibacterial**, `Antibacterial_amps_clean.fasta`, 5-60 aa; 200 épocas; instantáneas en las épocas 1, 20, 40, ..., 200; 2000 secuencias por instantánea; semilla 42; mismas métricas y mismos dos modos de decodificación, **sample** como principal y **argmax** como secundario) y se compara con los resultados de referencia ya generados, que **no se reentrenan**: se leen de `context/data/processed/`.

**Referencia de la Fase 3:** la GAN vanilla estabilizada con ruido de instancia constante (`gan_stabc`, sección h del notebook 02). Las otras dos variantes del 02 (`gan_base`, la vanilla pura, y `gan_stab`, la estabilizada con ruido decreciente) se muestran como contexto.

Contenido actual: **a. WGAN-GP** (Wasserstein GAN con penalización de gradiente) y **b. GAN condicional por longitud** (se le puede pedir la longitud del péptido; es el primero de los tres experimentos condicionales previstos: longitud, hidrofobicidad y ambas). Más adelante se añadirán aquí las demás mejoras como nuevas secciones c, d, ...

Orden: preparación, entorno, elección del backend y dispositivo de Keras (por defecto CPU; opcionalmente, una prueba de rendimiento), datos y, después, las secciones a y b (la b usa como comparación las variantes cargadas en la a).

> **Tiempo de ejecución aproximado.** La sección a (WGAN-GP) hace 169 pasos de crítico por época (cada uno con doble retropropagación), así que una época cuesta más que la de la vanilla. Por defecto el notebook usa la **CPU** con TensorFlow: unos **2.2 s por época, ~7 min** para las 200 épocas de la WGAN-GP. Con la celda opcional de la sección 2 (`EVALUAR_BACKEND = True`, requiere GPU) se usa el backend más rápido de la máquina: con torch en GPU (RTX 3060) son unos **1.1 s por época, ~4 min** (más ~40 s de la prueba de rendimiento). (Medidas en la máquina de desarrollo; torch en CPU: ~1.9 s por época.) La sección b (GAN condicional por longitud) tiene un coste por época parecido (algo mayor) al de la vanilla estabilizada: unos **4.5 s por época con TensorFlow en CPU (~15 min las 200 épocas)** y unos **2.6 s por época con torch en GPU (~9 min)**, sin contar las evaluaciones por instantánea (~8-11 s en total). Cargar las referencias y dibujar es inmediato. En total, el notebook entero tarda ~23 min en CPU y ~13 min con la GPU.
>
> **Reproducibilidad.** Todas las semillas están fijadas, pero los números pueden variar entre backends y dispositivos (las operaciones en coma flotante no dan resultados idénticos en TensorFlow, torch, CPU o GPU). Los CSV de referencia, los de `wgan_gp_*` y los de `cgan_length_*` (script `run_conditional_gan.py`) en `context/data/processed/` se generaron con **torch en GPU**; al ejecutar en CPU las curvas y métricas serán parecidas pero no idénticas, y los números concretos que cita el texto se generan a partir de la ejecución en curso. Es **una sola ejecución con la semilla 42**: no se estima la variabilidad entre semillas.

La lógica vive en `src/amp_gan/` (`wgan_gp`, `conditional_gan`, `conditions`, `experiment`, `evaluation`, `properties`, `runtime`); aquí solo se importa, se ejecuta y se dibuja. Las métricas son las del notebook 02: longitud, composición de aminoácidos, carga neta (pH 7,4) e hidrofobicidad media de Kyte-Doolittle (GRAVY).

## 0. Preparación: rutas, semilla y versiones

In [ ]:
import platform
import random
import sys
import time
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy

# Raíz del proyecto: se sube desde el directorio actual hasta encontrar `context/` y `src/` (sin rutas absolutas).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "context").is_dir() and (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))

# Importamos solo módulos que NO cargan Keras (el backend debe fijarse antes de importarlo).
from amp_gan import runtime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROCESSED = ROOT / "context" / "data" / "processed"
TRAIN_FASTA = PROCESSED / "Antibacterial_amps_clean.fasta"
BATCH_SIZE = 128
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})

print("python    :", platform.python_version())
print("numpy     :", np.__version__)
print("pandas    :", pd.__version__)
print("scipy     :", scipy.__version__)
print("matplotlib:", matplotlib.__version__)
print("semilla   :", SEED)
print("keras ya importado en este kernel:", "keras" in sys.modules)

## 1. Entorno

Versiones de las librerías de aprendizaje profundo (leídas de los metadatos del paquete, sin importarlas) y GPU que ve el sistema. Todavía no se ha importado Keras: su backend y su dispositivo se eligen en la sección siguiente.

In [ ]:
import importlib.metadata as md_
import os
import subprocess

def installed_version(*names):
    for name in names:
        try:
            return f"{md_.version(name)} (paquete {name})"
        except md_.PackageNotFoundError:
            pass
    return "no instalado"

for label, names in (("keras", ("keras", "keras-nightly")), ("tensorflow", ("tensorflow",)), ("torch", ("torch",))):
    print(f"{label:11s}: {installed_version(*names)}")
print("backends instalados para Keras:", runtime.installed_backends())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=20, check=True).stdout.strip()
    print("GPU (nvidia-smi):", gpu)
except Exception as exc:
    print("GPU (nvidia-smi): no disponible:", type(exc).__name__)

## 2. Backend y dispositivo de Keras

Keras 3 puede ejecutarse sobre TensorFlow o torch, en CPU o en GPU. Keras lee el backend (`KERAS_BACKEND`) y TensorFlow/torch leen la visibilidad de la GPU (`CUDA_VISIBLE_DEVICES`) **solo al importarse**, así que la elección debe hacerse antes de importar Keras (hasta aquí no se ha importado). Si el kernel ya hubiese importado Keras con otro backend, habría que reiniciarlo.

**Por defecto se usa la CPU**, con TensorFlow si está instalado (fue el más rápido en CPU) y, si no, torch. Así el notebook funciona en cualquier máquina, sin GPU y sin depender de archivos generados antes. Se puede forzar el backend con la variable de entorno `AMP_GAN_BACKEND` (`tensorflow` o `torch`) antes de arrancar el kernel.

La WGAN-GP funciona en **TensorFlow y en torch** (la penalización de gradiente necesita doble retropropagación, que está escrita por separado para cada backend); con JAX lanza un error claro.

In [ ]:
decision = runtime.apply_cpu_default()
assert "keras" not in sys.modules, "Keras ya está importado: reinicia el kernel antes de elegir el backend."
print(decision["message"])

### Opcional: ¿hay un backend/dispositivo más rápido en tu máquina?

> **Opcional: evaluar si hay un backend/dispositivo más rápido; si no dispones de gráfica puedes saltarla y se usa la CPU.**

Con `EVALUAR_BACKEND = True` la celda siguiente mide el paso de entrenamiento de la GAN base vanilla (mismo modelo, mismos datos, misma semilla; la WGAN-GP tiene un coste por época distinto, pero la decisión de backend se toma con esta misma prueba, que es la del resto del proyecto) para cada configuración `(backend, dispositivo)` **disponible en esta máquina** (solo se prueban los backends instalados y la GPU solo si se detecta) y varios tamaños de lote, y aplica la más rápida. Cada configuración se ejecuta en un subproceso independiente para que el backend y la GPU queden bien aislados, y una configuración que falle se descarta sin interrumpir el resto. Se miden menos pasos que en la prueba completa de `python -m amp_gan.benchmark` (60 pasos medidos y 20 de calentamiento, frente a 200 y 20), de modo que las cifras son algo más ruidosas; el resultado se devuelve en memoria y no se escribe a disco ni se lee ningún archivo de otra máquina.

Si no hay ninguna GPU utilizable, la prueba no se ejecuta y se mantiene la CPU. Con el valor por defecto (`False`) esta sección no hace nada más.

In [ ]:
EVALUAR_BACKEND = False  # True: medir y aplicar el backend/dispositivo más rápido (opcional; requiere GPU para que merezca la pena)

bench = None
if EVALUAR_BACKEND:
    t0 = time.perf_counter()
    decision, bench = runtime.apply_auto_backend(steps=60, warmup=20)
    print(f"Evaluación hecha en {time.perf_counter() - t0:.0f} s")
    print(decision["message"])
    if bench is not None:
        view = bench[["backend", "device", "batch_size", "status", "ms_per_step_median", "samples_per_s", "est_epoch_s", "error"]].copy()
        view["error"] = view["error"].fillna("").str.slice(0, 90)
        display(view.round(3).reset_index(drop=True))
else:
    print("Evaluación omitida (EVALUAR_BACKEND = False): se mantiene la configuración por defecto.")
    print(decision["message"])

Solo si se ha evaluado: tiempo estimado por época (segundos; menos es mejor) por configuración y tamaño de lote. Las configuraciones que fallan no tienen barra: se marcan como *no disponible* en la leyenda y su motivo se resume debajo del gráfico. Las configuraciones no disponibles en la máquina (por ejemplo, TensorFlow en GPU si no se detecta) ni siquiera se prueban.

In [ ]:
if bench is not None:
    configs = list(dict.fromkeys(zip(bench["backend"], bench["device"])))
    batch_sizes = sorted(bench["batch_size"].unique())
    colors = {("tensorflow", "cpu"): "tab:blue", ("torch", "cpu"): "tab:green",
              ("torch", "gpu"): "tab:red", ("tensorflow", "gpu"): "tab:purple"}

    fig, ax = plt.subplots(figsize=(10, 4.2))
    width = 0.8 / len(configs)
    for i, (backend, device) in enumerate(configs):
        sub = bench[(bench["backend"] == backend) & (bench["device"] == device)].set_index("batch_size")
        ok = (sub["status"] == "ok").any()
        label = f"{backend}/{device}" + ("" if ok else " (no disponible)")
        x = np.arange(len(batch_sizes)) + (i - (len(configs) - 1) / 2) * width
        vals = [sub.loc[b, "est_epoch_s"] if b in sub.index and sub.loc[b, "status"] == "ok" else np.nan for b in batch_sizes]
        bars = ax.bar(x, np.nan_to_num(vals), width=width, color=colors.get((backend, device), "gray"), label=label)
        for rect, v in zip(bars, vals):
            if np.isfinite(v):
                ax.text(rect.get_x() + rect.get_width() / 2, v, f"{v:.1f}", ha="center", va="bottom", fontsize=7)
    ax.set_xticks(np.arange(len(batch_sizes)))
    ax.set_xticklabels(batch_sizes)
    ax.set_xlabel("Tamaño de lote")
    ax.set_ylabel("Tiempo estimado por época (s)")
    ax.set_title("Prueba de rendimiento de la GAN base: tiempo por época")
    ax.legend(title="backend/dispositivo")
    fig.tight_layout()
    plt.show()

    errors = bench[bench["status"] != "ok"].drop_duplicates(["backend", "device"])
    for _, r in errors.iterrows():
        print(f"No disponible: {r['backend']}/{r['device']} -> {str(r['error'])[:200]}")

La regla de decisión del proyecto, para el tamaño de lote de entrenamiento (128), es: gana la configuración con menor tiempo estimado por época; si la segunda está a menos de un 10 % se declara empate y se prefiere la opción más simple (CPU antes que GPU, TensorFlow antes que torch). El backend y el dispositivo ya están fijados en el entorno, **antes de importar Keras**; la celda siguiente muestra la configuración vigente.

In [ ]:
assert "keras" not in sys.modules, "Keras ya está importado: reinicia el kernel antes de elegir el backend."
print(f"Configuración aplicada: modo {decision['mode']}, backend={decision['backend']}, dispositivo={decision['device']}")

Ahora sí se importa Keras (a través de `describe_environment`) y se confirma el backend y el dispositivo realmente usados.

In [ ]:
env = runtime.describe_environment()
for k in ("keras_backend", "keras_devices", "gpu_visible", "cuda_visible_devices", "keras_version"):
    print(f"{k:22s}: {env[k]}")
assert env["keras_backend"] == decision["backend"]
assert env["gpu_visible"] == (decision["device"] == "gpu")

## 3. Datos

Se cargan las secuencias limpias de Antibacterial (5-60 aa, 20 aminoácidos estándar, sin duplicados exactos) y se codifican en one-hot con relleno `PAD` hasta 60 posiciones.

In [ ]:
from amp_gan.encoding import SEQ_LEN, VOCAB_SIZE, load_encoded

X, df = load_encoded(TRAIN_FASTA)
print(f"secuencias: {len(df):,}")
print(f"X: forma {X.shape}, dtype {X.dtype}, tamaño {X.nbytes / 2**20:.0f} MiB")
print(f"seq_len = {SEQ_LEN}, vocabulario = {VOCAB_SIZE} (20 aa + PAD)")

## a. WGAN-GP

### a.1 Qué cambia frente a la GAN vanilla

La vanilla pura del notebook 02 colapsa: el discriminador gana desde las primeras épocas (su pérdida cae a ~1e-7) y el generador acaba produciendo pocas secuencias distintas. La **WGAN-GP** (Gulrajani et al., 2017) sustituye el planteamiento de clasificación real/falso por la estimación de una distancia:

- **Distancia de Wasserstein.** El discriminador pasa a ser un **crítico**: da una puntuación real (sin sigmoide, última capa lineal) y se entrena para que la diferencia entre la puntuación media de las secuencias reales y la de las generadas sea lo mayor posible. Esa diferencia estima la distancia de Wasserstein-1 entre las dos distribuciones y, a diferencia de la entropía cruzada, sigue dando gradiente útil al generador aunque el crítico separe bien las dos clases.
- **Pérdidas.** Crítico: `mean(D(falso)) - mean(D(real)) + λ · GP`. Generador: `-mean(D(G(z)))`.
- **Penalización de gradiente (GP).** La estimación de Wasserstein exige que el crítico sea 1-Lipschitz. En lugar de recortar pesos, se penaliza que la norma del gradiente del crítico se aleje de 1 en puntos interpolados entre real y generado: `GP = mean((‖∇x̂ D(x̂)‖₂ - 1)²)` con `x̂ = ε·real + (1-ε)·generado` y `ε ~ U(0, 1)` por muestra. Derivar respecto de la **entrada** del crítico y volver a derivar (doble retropropagación) es lo único del proyecto que no se puede escribir solo con `keras.ops`; en `src/amp_gan/wgan_gp.py` está aislado en funciones pequeñas para TensorFlow (`tf.GradientTape` anidada) y torch (`torch.autograd.grad(create_graph=True)`).
- **5 pasos de crítico por cada paso del generador** (`n_critic = 5`), con λ = 10, Adam de tasa de aprendizaje 1e-4 y β = (0; 0,9). Son los valores canónicos del artículo: **no se ajusta ningún hiperparámetro**.
- **Sin las correcciones de la estabilizada.** No hay suavizado de etiquetas (no hay etiquetas), ni ruido de instancia, ni Gumbel-softmax, ni Dropout ni BatchNorm en el crítico (la penalización por muestra no es compatible con BatchNorm). El generador entrega probabilidades softmax y el crítico compara las secuencias reales (one-hot) con esas probabilidades, como en el artículo original.
- **Mismas redes que el resto:** el generador es exactamente el de las demás GAN (`build_generator`, MLP 256-512 con softmax por posición); el crítico es `Flatten` → `Dense(512)` + LeakyReLU → `Dense(256)` + LeakyReLU → `Dense(1)`, con la misma anchura que el discriminador.

### Definición de «época» en este notebook

Una **época** es una pasada por los datos reales: `n // batch_size` = **169 pasos de crítico**, cada uno con un lote real distinto (se descarta el último lote incompleto, como en las otras GAN). El generador se actualiza **una vez cada 5 pasos de crítico**: unas **33-34 actualizaciones del generador por época**, frente a las 169 de las GAN anteriores (una por paso). Por tanto, en 200 épocas el generador de la WGAN-GP recibe unas 6 760 actualizaciones y el de las otras, 33 800, aunque el crítico ve los mismos datos las mismas veces. El historial registra `critic_steps` y `generator_steps` de cada época.

### Criterio de «mejora» fijado de antemano

Se evalúa en el modo **sample**, en la **época 200** (sin elegir la mejor época). La WGAN-GP **mejora a la referencia** (estabilizada con ruido constante, `gan_stabc`) si **cumple las cinco condiciones**:

1. reduce la W1 de longitud;
2. reduce la W1 de carga neta;
3. reduce la W1 de hidrofobicidad;
4. reduce la variación total de composición;
5. mantiene `unique_fraction` ≥ 0,95.

Se informa de cada condición por separado y del resultado global, sea cual sea. No se ajustan hiperparámetros ni se repite con otra semilla.

### a.2 Entrenamiento

Se usa `WGANConfig()` y el mismo `run_gan_experiment` que en el notebook 02 (con `train_fn=train_wgan_gp`): 200 épocas, instantáneas cada 20 (y la primera), 2000 secuencias por instantánea con un ruido fijo, semilla 42. La primera época incluye la construcción (y, con TensorFlow, la compilación) del modelo; el tiempo de entrenamiento no incluye el de las evaluaciones.

In [ ]:
from amp_gan.experiment import MODES, run_gan_experiment
from amp_gan.wgan_gp import WGANConfig, train_wgan_gp

EPOCHS, SNAPSHOT_EVERY, N_EVAL = 200, 20, 2000
config_w = WGANConfig()
print(config_w)

res_w = run_gan_experiment(X, df, config_w, epochs=EPOCHS, snapshot_every=SNAPSHOT_EVERY, n_eval=N_EVAL,
                           eval_seed=SEED, train_fn=train_wgan_gp)
t_w = res_w.timing
print(f"Tiempo total: {t_w['total_s']:.1f} s (entrenamiento {t_w['train_s']:.1f} s + evaluaciones {t_w['eval_s']:.1f} s)")
print(f"Por época: media {t_w['mean_epoch_s']:.2f} s, mediana {t_w['median_epoch_s']:.2f} s "
      f"(backend {env['keras_backend']}, dispositivo {decision['device']})")

### a.3 Curvas de pérdida

Cuatro medidas por época. **Crítico:** pérdida total (negativa cuando el crítico separa lo real de lo generado; incluye la penalización). **Estimación de Wasserstein:** `mean(D(real)) - mean(D(falso))`, es decir, menos la pérdida del crítico sin penalización; baja a medida que el generador se acerca a los datos. **Penalización de gradiente:** `GP` sin multiplicar por λ (0 = el gradiente del crítico tiene norma 1 en todas partes). **Generador:** `-mean(D(G(z)))`, promediada sobre las ~33 actualizaciones del generador de cada época. A diferencia de las GAN anteriores, estas pérdidas no están acotadas por ln 2 ni se interpretan como probabilidades.

In [ ]:
h_w = res_w.history
epochs_axis = np.arange(1, EPOCHS + 1)
panels_loss = [("critic_loss", "Pérdida del crítico (con GP)", "tab:blue"),
               ("wasserstein_estimate", "Estimación de Wasserstein", "tab:purple"),
               ("gradient_penalty", "Penalización de gradiente (sin λ, log.)", "tab:orange"),
               ("g_loss", "Pérdida del generador", "tab:green")]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, (key, title, color) in zip(axes, panels_loss):
    ax.plot(epochs_axis, h_w[key], color=color)
    ax.set_title(title)
    ax.set_xlabel("Época")
axes[2].set_yscale("log")
axes[1].axhline(0, color="gray", ls=":", lw=1)
fig.tight_layout()
plt.show()

hist_w = pd.DataFrame({k: h_w[k] for k in ("critic_loss", "wasserstein_estimate", "gradient_penalty", "g_loss",
                                            "critic_steps", "generator_steps")}, index=epochs_axis)
display(pd.DataFrame({"primeras 5 épocas": hist_w.head(5).mean(), "épocas 96-100": hist_w.loc[96:100].mean(),
                      "últimas 5 épocas": hist_w.tail(5).mean()}).round(4))
print(f"Pasos de crítico por época: {hist_w['critic_steps'].iloc[0]}; actualizaciones del generador por época: "
      f"{hist_w['generator_steps'].min()}-{hist_w['generator_steps'].max()} "
      f"(total {int(hist_w['generator_steps'].sum()):,} en {EPOCHS} épocas).")
print("¿Algún valor no finito en el historial?", bool((~np.isfinite(hist_w.to_numpy(dtype=float))).any()))

### a.4 Resultados de referencia (sin reentrenar)

Se cargan de `context/data/processed/` las métricas, el historial, la información de la ejecución y las 2000 secuencias de la época final de las tres variantes del notebook 02. Si falta algún archivo, la celda se detiene con el comando que lo genera. Estas ejecuciones se hicieron con torch en GPU (ver `run_info.json`).

In [ ]:
import json

from amp_gan.data import read_fasta

REF_NAME = "estab. ruido constante (referencia)"
REFERENCIAS = {"vanilla pura": "gan_base", "estabilizada": "gan_stab", REF_NAME: "gan_stabc"}
VARIANTE_DE = {"gan_base": "base", "gan_stab": "stabilized", "gan_stabc": "stabilized_const"}
SUFIJOS = ("metrics.csv", "history.csv", "run_info.json", "generated_sample.fasta", "generated_argmax.fasta")

faltan = [f"{p}_{s}" for p in REFERENCIAS.values() for s in SUFIJOS if not (PROCESSED / f"{p}_{s}").exists()]
if faltan:
    prefijos = sorted({"_".join(f.split("_")[:2]) for f in faltan})
    comandos = "\n".join(f"    python src/amp_gan/run_gan_base.py --variant {VARIANTE_DE[p]}" for p in prefijos)
    raise FileNotFoundError(
        "Faltan resultados de referencia en context/data/processed/ (" + ", ".join(faltan) + ").\n"
        "Genéralos desde la raíz del proyecto con:\n" + comandos)

ref_metrics, ref_hist, ref_info, ref_seqs = {}, {}, {}, {}
for name, prefix in REFERENCIAS.items():
    m = pd.read_csv(PROCESSED / f"{prefix}_metrics.csv")
    ref_metrics[name] = m[m["epoch"].notna()].astype({"epoch": int})  # las filas de suelo no tienen época
    ref_hist[name] = pd.read_csv(PROCESSED / f"{prefix}_history.csv")
    ref_info[name] = json.loads((PROCESSED / f"{prefix}_run_info.json").read_text(encoding="utf-8"))
    ref_seqs[name] = {mode: read_fasta(PROCESSED / f"{prefix}_generated_{mode}.fasta")["sequence"].tolist() for mode in MODES}

# Las cuatro variantes, en el orden de las tablas: las tres de referencia y la WGAN-GP de esta ejecución.
ALL_METRICS = {**ref_metrics, "WGAN-GP": res_w.metrics}
ALL_SEQS = {**ref_seqs, "WGAN-GP": {mode: res_w.sequences[mode] for mode in MODES}}
COLORS = {"vanilla pura": "tab:red", "estabilizada": "tab:green", REF_NAME: "tab:blue", "WGAN-GP": "tab:purple"}

resumen = []
for name, prefix in REFERENCIAS.items():
    info, hh = ref_info[name], ref_hist[name]
    steps = info["n_train"] // info["config"]["batch_size"]
    resumen.append({"variante": name, "archivos": prefix + "_*", "épocas": len(hh), "backend": info["keras_backend"],
                    "dispositivo": info["device_used"], "s/época": info["timing_s"]["mean_epoch_s"],
                    "actualizaciones de G por época": steps})
resumen.append({"variante": "WGAN-GP (esta ejecución)", "archivos": "-", "épocas": EPOCHS, "backend": env["keras_backend"],
                "dispositivo": decision["device"], "s/época": t_w["mean_epoch_s"],
                "actualizaciones de G por época": float(hist_w["generator_steps"].mean())})
display(pd.DataFrame(resumen).set_index("variante").round(2))

### a.5 Métricas finales (época 200)

Métricas de la época final en cada modo de decodificación para las cuatro variantes y los dos suelos (`real_subset`: lo mínimo alcanzable por el ruido del muestreo; `random_uniform`: ningún aprendizaje). Mismas columnas y definiciones que en el notebook 02: fracción de secuencias válidas, únicas y nuevas; media, desviación típica y W1 (distancia de Wasserstein-1 frente a las secuencias reales) de longitud, carga neta e hidrofobicidad; y `composition_tv`, la variación total entre la composición media de aminoácidos generada y la real.

In [ ]:
from amp_gan.evaluation import is_valid, profile_sequences, reference_floors
from amp_gan.properties import properties_table

floors = reference_floors(df, n=N_EVAL, seed=SEED)
props_real = properties_table(df)
cols = ["valid_fraction", "unique_fraction", "novel_fraction", "length_mean", "length_std", "length_wasserstein",
        "net_charge_mean", "net_charge_std", "net_charge_wasserstein",
        "hydrophobicity_mean", "hydrophobicity_std", "hydrophobicity_wasserstein", "composition_tv"]

def pick(metrics_df, mode, epoch=EPOCHS):
    return metrics_df[(metrics_df["epoch"] == epoch) & (metrics_df["mode"] == mode)].iloc[0]

def final_table(mode):
    table = pd.DataFrame({name: pick(m, mode) for name, m in ALL_METRICS.items()}
                         | {"suelo real_subset": pd.Series(floors["real_subset"]),
                            "suelo random_uniform": pd.Series(floors["random_uniform"])})
    return table.loc[cols]

print("Referencia real (Antibacterial completo):")
display(pd.DataFrame({"media": props_real[["length", "net_charge", "hydrophobicity"]].mean(),
                      "desv. típica": props_real[["length", "net_charge", "hydrophobicity"]].std()}).round(3))
print("Modo sample (principal), época final:")
display(final_table("sample").round(3))
print("Modo argmax (secundario), época final:")
display(final_table("argmax").round(3))

### a.6 Distribuciones: real, referencia y WGAN-GP (época final)

Histogramas normalizados a fracción de secuencias, con los mismos intervalos y ejes para las tres series en cada panel (gris: real; azul: referencia, estabilizada con ruido constante; morado: WGAN-GP). Arriba, modo sample; abajo, modo argmax. Solo se usan las secuencias generadas válidas.

In [ ]:
valid_seqs = {name: {mode: [s for s in seqs[mode] if is_valid(s)] for mode in MODES} for name, seqs in ALL_SEQS.items()}
prof_gen = {name: {mode: profile_sequences(v) if v else None for mode, v in d.items()} for name, d in valid_seqs.items()}
prof_real = profile_sequences(df["sequence"].tolist())

PROPS = [("length", "Longitud (aa)"), ("net_charge", "Carga neta (pH 7,4)"), ("hydrophobicity", "Hidrofobicidad (GRAVY)")]
SHOWN = [REF_NAME, "WGAN-GP"]

edges = {}
for prop, _ in PROPS:
    allv = np.concatenate([prof_real.values(prop)] + [prof_gen[n][m].values(prop) for n in SHOWN for m in MODES if prof_gen[n][m] is not None])
    lo, hi = np.floor(allv.min()), np.ceil(allv.max())
    edges[prop] = np.arange(lo - 0.5, hi + 1.5) if prop == "length" else np.linspace(lo, hi, 41)

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex="col", sharey="col")
for r, mode in enumerate(("sample", "argmax")):
    for c, (prop, xlabel) in enumerate(PROPS):
        ax = axes[r, c]
        real = prof_real.values(prop)
        ax.hist(real, bins=edges[prop], weights=np.full(real.size, 1 / real.size), color="gray", alpha=0.4, label=f"real (n={real.size:,})")
        for name in SHOWN:
            p = prof_gen[name][mode]
            if p is not None:
                g = p.values(prop)
                ax.hist(g, bins=edges[prop], weights=np.full(g.size, 1 / g.size), histtype="step", lw=1.6, color=COLORS[name],
                        label=f"{name} (n={g.size:,})")
        ax.set_title(f"{xlabel} - {mode}")
        ax.set_xlabel(xlabel)
        if c == 0:
            ax.set_ylabel("Fracción de secuencias")
        ax.legend(fontsize=7, loc="upper right")
for c in range(3):  # eje vertical común por columna, con hueco para la leyenda
    axes[0, c].set_ylim(0, max(axes[r, c].get_ylim()[1] for r in range(2)) * 1.3)
fig.tight_layout()
plt.show()

### a.7 Composición de aminoácidos

Fracción media de cada aminoácido (media de las fracciones por secuencia) en las secuencias reales y en las generadas válidas de la referencia y de la WGAN-GP, ordenados por abundancia en las reales, con el mismo eje vertical en los dos modos.

In [ ]:
from amp_gan.data import STANDARD_ALPHABET

comp = pd.DataFrame({"real": prof_real.composition}, index=list(STANDARD_ALPHABET))
for mode in MODES:
    for name in SHOWN:
        comp[f"{name}|{mode}"] = prof_gen[name][mode].composition if prof_gen[name][mode] is not None else np.nan
comp = comp.sort_values("real", ascending=False)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, sharey=True)
x = np.arange(len(comp))
w = 0.27
for ax, mode in zip(axes, ("sample", "argmax")):
    series = [("real", "real", "gray")] + [(f"{n}|{mode}", n, COLORS[n]) for n in SHOWN]
    for i, (col, label, color) in enumerate(series):
        ax.bar(x + (i - 1) * w, comp[col] * 100, width=w, color=color, label=label)
    ax.set_ylabel("Composición media (%)")
    ax.set_title(f"Composición media de aminoácidos - {mode} (época final)")
    ax.legend()
axes[-1].set_xticks(x)
axes[-1].set_xticklabels(comp.index)
axes[-1].set_xlabel("Aminoácido (ordenados por abundancia en las secuencias reales)")
fig.tight_layout()
plt.show()

### a.8 Evolución de las métricas por instantánea

Las mismas métricas que en el notebook 02 para las cuatro variantes (continua: sample; discontinua: argmax). La línea gris punteada es el suelo `real_subset` (en la longitud media, la media real). La vanilla pura queda muy lejos en varias métricas (colapsa) y puede comprimir la escala de las demás.

In [ ]:
from matplotlib.lines import Line2D

panels = [
    ("unique_fraction", "Fracción de secuencias únicas", None),
    ("length_wasserstein", "W1 de longitud (aa)", None),
    ("net_charge_wasserstein", "W1 de carga neta", None),
    ("hydrophobicity_wasserstein", "W1 de hidrofobicidad", None),
    ("composition_tv", "Variación total de composición", None),
    ("length_mean", "Longitud media (aa)", float(props_real["length"].mean())),
]
fig, axes = plt.subplots(2, 3, figsize=(14, 7.6))
for ax, (col, title, real_value) in zip(axes.ravel(), panels):
    for name, m in ALL_METRICS.items():
        for mode, ls in (("sample", "-"), ("argmax", "--")):
            sel = m[m["mode"] == mode]
            ax.plot(sel["epoch"], sel[col], ls, marker="o", ms=3.5, color=COLORS[name])
    ax.axhline(real_value if real_value is not None else floors["real_subset"][col], color="gray", ls=":")
    ax.set_title(title)
    ax.set_xlabel("Época")
    ax.set_ylabel(title)
handles = [Line2D([], [], color=COLORS[n_], ls=ls_, marker="o", ms=3.5, label=f"{n_}, {m_}")
           for n_ in ALL_METRICS for m_, ls_ in (("sample", "-"), ("argmax", "--"))]
handles.append(Line2D([], [], color="gray", ls=":", label="suelo real_subset (media real en la longitud media)"))
fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=8, frameon=False)
fig.tight_layout(rect=(0, 0.1, 1, 1))
plt.show()

### W1 de longitud y longitud media por instantánea (modo sample)

Valores numéricos de las curvas de longitud de las cuatro variantes. La longitud media real se indica en el título de la segunda tabla.

In [ ]:
def snap_series(metrics_df, col, mode="sample"):
    return metrics_df[metrics_df["mode"] == mode].set_index("epoch")[col]

real_mean_len = float(props_real["length"].mean())
snap = pd.concat({
    "W1 longitud": pd.DataFrame({n: snap_series(m, "length_wasserstein") for n, m in ALL_METRICS.items()}),
    f"longitud media (real = {real_mean_len:.2f})": pd.DataFrame({n: snap_series(m, "length_mean") for n, m in ALL_METRICS.items()}),
}, axis=1)
snap.index.name = "época"
print(f"Suelo real_subset de la W1 de longitud: {floors['real_subset']['length_wasserstein']:.3f}")
display(snap.round(3))

### a.9 Secuencias de ejemplo y repetición

Las diez primeras secuencias de la WGAN-GP en cada modo (época final) con su longitud, carga neta e hidrofobicidad (las no válidas aparecen sin propiedades). Debajo, los recuentos de repetición de las cuatro variantes: secuencias válidas, distintas entre ellas y veces que aparece la más repetida.

In [ ]:
from collections import Counter

def example_table(seqs, k=10):
    rows = []
    for s in seqs[:k]:
        if is_valid(s):
            p = profile_sequences([s])
            rows.append({"sequence": s, "length": len(s), "net_charge": p.net_charge[0], "hydrophobicity": p.hydrophobicity[0]})
        else:
            rows.append({"sequence": s if s else "(vacía)", "length": len(s), "net_charge": np.nan, "hydrophobicity": np.nan})
    return pd.DataFrame(rows).round(2)

pd.set_option("display.max_colwidth", 80)
for mode in ("sample", "argmax"):
    print(f"WGAN-GP, modo {mode}:")
    display(example_table(ALL_SEQS["WGAN-GP"][mode]))

rep_rows = []
for name, d in ALL_SEQS.items():
    for mode in ("sample", "argmax"):
        valid = [s for s in d[mode] if is_valid(s)]
        cnt = Counter(valid)
        top_seq, top_n = cnt.most_common(1)[0] if cnt else ("", 0)
        rep_rows.append({"variante": name, "modo": mode, "generadas": len(d[mode]), "válidas": len(valid),
                         "distintas": len(cnt), "más repetida (veces)": top_n, "secuencia más repetida": top_seq[:40]})
print("Recuentos de repetición (época final):")
display(pd.DataFrame(rep_rows).set_index(["variante", "modo"]))

### a.10 Conclusiones (descriptivas)

Primero se evalúa el criterio fijado de antemano (modo sample, época 200, sin elegir la mejor época), condición por condición, con las cifras de esta ejecución. Después se describen, sin interpretar causas, las demás métricas, la evolución de la longitud, los tiempos y el comportamiento del entrenamiento. El texto se genera con el código de la celda siguiente.

In [ ]:
ref_s, wg_s = pick(ALL_METRICS[REF_NAME], "sample"), pick(ALL_METRICS["WGAN-GP"], "sample")
UNIQUE_MIN = 0.95
condiciones = [
    ("W1 de longitud", "length_wasserstein"),
    ("W1 de carga neta", "net_charge_wasserstein"),
    ("W1 de hidrofobicidad", "hydrophobicity_wasserstein"),
    ("variación total de composición", "composition_tv"),
]
print("Criterio fijado de antemano (modo sample, época 200): la WGAN-GP mejora a la referencia (estabilizada con ruido constante) si")
print(f"reduce la W1 de longitud, la W1 de carga neta, la W1 de hidrofobicidad y la variación total de composición, y mantiene unique_fraction >= {UNIQUE_MIN}.\n")
cumple = []
for label, col in condiciones:
    ok = bool(wg_s[col] < ref_s[col])
    cumple.append(ok)
    rel = (wg_s[col] - ref_s[col]) / ref_s[col]
    print(f"  - {label}: referencia {ref_s[col]:.3f}, WGAN-GP {wg_s[col]:.3f} ({rel:+.0%}) -> {'SE CUMPLE' if ok else 'NO se cumple'} (se pedía reducir).")
ok_u = bool(wg_s["unique_fraction"] >= UNIQUE_MIN)
cumple.append(ok_u)
print(f"  - unique_fraction: referencia {ref_s['unique_fraction']:.3f}, WGAN-GP {wg_s['unique_fraction']:.3f} -> {'SE CUMPLE' if ok_u else 'NO se cumple'} (se pedía >= {UNIQUE_MIN}).")
print(f"\nCondiciones cumplidas: {sum(cumple)} de {len(cumple)}. Según el criterio, la WGAN-GP {'SÍ' if all(cumple) else 'NO'} mejora a la referencia.")
print("Es una sola ejecución con la semilla 42: no se estima la variabilidad entre semillas ni se ajusta ningún hiperparámetro.")

print()
fl, ru = floors["real_subset"], floors["random_uniform"]
print(f"Referencia real: longitud {props_real['length'].mean():.1f} +- {props_real['length'].std():.1f}, "
      f"carga {props_real['net_charge'].mean():.2f} +- {props_real['net_charge'].std():.2f}, "
      f"GRAVY {props_real['hydrophobicity'].mean():.2f} +- {props_real['hydrophobicity'].std():.2f}.")
for mode in ("sample", "argmax"):
    mw, mr = pick(ALL_METRICS["WGAN-GP"], mode), pick(ALL_METRICS[REF_NAME], mode)
    print(f"\n[{mode}] época {EPOCHS}: WGAN-GP (referencia entre paréntesis; suelos real_subset / random_uniform)")
    for col, label in (("valid_fraction", "válidas"), ("unique_fraction", "únicas"), ("novel_fraction", "nuevas"),
                       ("length_mean", "longitud media"), ("length_wasserstein", "W1 longitud"),
                       ("net_charge_mean", "carga media"), ("net_charge_wasserstein", "W1 carga"),
                       ("hydrophobicity_mean", "GRAVY medio"), ("hydrophobicity_wasserstein", "W1 hidrofobicidad"),
                       ("composition_tv", "variación total de composición")):
        suelos = f"{fl[col]:.3f} / {ru[col]:.3f}" if col in fl else "-"
        print(f"  - {label}: {mw[col]:.3f} ({mr[col]:.3f}); suelos {suelos}")

print()
w1 = snap_series(ALL_METRICS["WGAN-GP"], "length_wasserstein")
lm = snap_series(ALL_METRICS["WGAN-GP"], "length_mean")
print("W1 de longitud por instantánea (sample), WGAN-GP: " + "; ".join(f"{e}: {v:.2f}" for e, v in w1.items()))
print("Longitud media por instantánea (sample), WGAN-GP: " + "; ".join(f"{e}: {v:.1f}" for e, v in lm.items()) + f" (real {real_mean_len:.1f}).")
wr = snap_series(ALL_METRICS[REF_NAME], "length_wasserstein")
print(f"Entre las épocas 20 y {EPOCHS}, la W1 de longitud de la WGAN-GP va de {w1[w1.index >= 20].max():.2f} (máximo) a {w1[w1.index >= 20].min():.2f} (mínimo); "
      f"la de la referencia, de {wr[wr.index >= 20].max():.2f} a {wr[wr.index >= 20].min():.2f}.")

f5, l5 = hist_w.head(5).mean(), hist_w.tail(5).mean()
print(f"\nPérdidas medias (primeras 5 -> últimas 5 épocas): crítico {f5['critic_loss']:.3f} -> {l5['critic_loss']:.3f}; "
      f"estimación de Wasserstein {f5['wasserstein_estimate']:.3f} -> {l5['wasserstein_estimate']:.3f}; "
      f"penalización {f5['gradient_penalty']:.4f} -> {l5['gradient_penalty']:.4f}; generador {f5['g_loss']:.3f} -> {l5['g_loss']:.3f}.")
rc = pd.DataFrame(rep_rows)
for name in ("WGAN-GP", REF_NAME):
    sub = rc[rc["variante"] == name]
    print(f"{name}: " + "; ".join(f"[{d['modo']}] {d['distintas']} secuencias distintas de {d['válidas']} válidas "
                                    f"(la más repetida, {d['más repetida (veces)']} veces)" for _, d in sub.iterrows()) + ".")
n_gu = int(hist_w["generator_steps"].sum())
print(f"Actualizaciones del generador: {n_gu:,} en total ({hist_w['generator_steps'].mean():.1f} por época) frente a "
      f"{ref_info[REF_NAME]['n_train'] // ref_info[REF_NAME]['config']['batch_size'] * EPOCHS:,} en la referencia.")
print(f"Tiempo: WGAN-GP {t_w['mean_epoch_s']:.2f} s/época de media ({t_w['total_s']:.0f} s en total, backend {env['keras_backend']} "
      f"en {decision['device']}); referencia {ref_info[REF_NAME]['timing_s']['mean_epoch_s']:.2f} s/época "
      f"({ref_info[REF_NAME]['keras_backend']} en {ref_info[REF_NAME]['device_used']}).")

### a.11 Nota sobre tiempos

El coste por época de la WGAN-GP es mayor que el de la vanilla porque cada paso de crítico calcula la penalización con doble retropropagación (y hay 169 pasos de crítico por época). Los tiempos medidos en la máquina de desarrollo (RTX 3060 portátil y CPU de 16 hilos; 200 épocas, sin contar las evaluaciones) están en la introducción; la celda anterior imprime los de esta ejecución. Con TensorFlow, el paso del crítico y el del generador se compilan con `tf.function` (sin compilar, una época tardaba ~4 veces más en CPU).

## b. GAN condicional por longitud

### b.1 Qué cambia y por qué

Hasta ahora el generador recibe solo ruido `z` y no se puede pedir nada concreto: la longitud de lo que sale la decide el azar del entrenamiento. La **GAN condicional** (Mirza y Osindero, 2014) añade una **condición numérica** `c` que se da al generador y al discriminador, de modo que se puede pedir *«un péptido de longitud L»*. Es el primero de tres experimentos previstos (longitud, hidrofobicidad y ambas a la vez); aquí solo se **ejecuta el de longitud**, aunque `src/amp_gan/conditions.py` y `src/amp_gan/conditional_gan.py` ya admiten hidrofobicidad y las dos condiciones juntas.

**Base:** la vanilla estabilizada con ruido de instancia constante (`GANConfig.stabilized_constant_noise()`, la referencia de la Fase 3): etiqueta real 0,9, ruido de instancia constante de desviación 0,1, Gumbel-softmax *straight-through* con τ = 1, generador MLP (256, 512), discriminador MLP (512, 256), `z` de 100, Adam con tasa 2e-4 y β1 = 0,5, lote 128, semilla 42. **No se cambia ningún hiperparámetro.** Solo se añade la condición:

- **Condición.** La longitud se normaliza a `[0, 1]` con `(L - 5) / 55` (rango 5-60 aa del proyecto). Para hidrofobicidad, la normalización sería min-max con el mínimo y el máximo del conjunto de entrenamiento.
- **Generador `[z, c]`.** La condición normalizada se **concatena a `z`** a la entrada del primer `Dense` (101 entradas); el resto del MLP es idéntico.
- **Discriminador `[secuencia, c]`.** La condición se **concatena a la secuencia aplanada** (1 261 entradas): el ruido de instancia se aplica a la secuencia y no a la condición; el resto es idéntico.
- **Modelo combinado** con entradas `[z, c]`: `D(Gumbel(G([z, c])), c)`.
- **Entrenamiento.** Igual que en la vanilla, pero cada lote lleva la **condición real** de sus secuencias (la longitud verdadera, normalizada), que se usa tanto para el lote real como para generar el falso y para el paso del generador: el generador recibe siempre las mismas condiciones que el lote real. La condición se entrena con **todas** las longitudes (5-60 aa).
- **Generación.** Se pide una condición y se decodifica con `sample_sequences` (modos `sample` y `argmax`, como siempre).

### Rango «habitual» y longitudes objetivo

Lo que se pide al generar y se evalúa es el **rango habitual de longitud: percentiles 10 a 90 de las secuencias reales de Antibacterial**. Se evalúan cinco longitudes objetivo, los percentiles 10, 30, 50, 70 y 90 (redondeados al entero más cercano). La celda siguiente los calcula de los datos.

### Dos evaluaciones

- **(i) Evaluación «empírica».** Las condiciones se toman de longitudes reales (muestreo con reposición de las longitudes del conjunto de entrenamiento, semilla 42): las métricas tienen el **mismo formato** que las de las demás variantes y son comparables con la referencia sin condicionar.
- **(ii) Evaluación por objetivos.** Para cada longitud objetivo `t` se piden 2000 secuencias con esa longitud y se mide la longitud media generada, el error absoluto medio `|L - t|`, la **tasa de acierto** (fracción con `|L - t| ≤ 2`; tolerancia fijada de antemano), la fracción de válidas y de únicas y la W1 de carga neta e hidrofobicidad frente a los datos reales con longitud en `[t - 2, t + 2]` (si ese subconjunto tiene menos de 30 secuencias, se indica y la W1 queda en NaN). La longitud se mide sobre todas las secuencias generadas, también las no válidas (una cadena vacía cuenta con longitud 0). Referencias **sin condición** para esta evaluación: (a) la muestra de la referencia (`gan_stabc`, 2000 secuencias), que no se puede dirigir: tasa de acierto ±2 para cada `t` y longitud media de toda la muestra; (b) el **techo real**: fracción de los datos reales con `|L - t| ≤ 2`.

### Criterios fijados de antemano (modo sample, época 200, sin elegir la mejor época)

**A. Control de la longitud.** La condicional controla la longitud si cumple las tres condiciones:
1. la pendiente de la regresión lineal de la longitud media generada frente a la longitud pedida (los cinco objetivos) está entre 0,8 y 1,2;
2. el error absoluto medio `|L - t|` promediado sobre los objetivos es ≤ 3 aa;
3. su tasa de acierto (±2) supera a la de la referencia en los cinco objetivos.

**B. Fidelidad con condiciones reales** (evaluación (i)). La condicional **no empeora a la referencia** si su W1 de longitud es ≤ que la de la referencia (3,854 en la época 200) y su `unique_fraction` es ≥ 0,95. Las W1 de carga neta e hidrofobicidad y la variación total de composición se informan por separado y de forma descriptiva (sin condición de aprobado), junto a las de la referencia.

Se informa de cada condición por separado y del resultado global de A y de B, sea cual sea; no se ajustan hiperparámetros ni se repite con otra semilla. Es **una sola ejecución con la semilla 42**.

In [ ]:
from amp_gan.conditions import (HIT_TOLERANCE, TARGET_PERCENTILES, length_control_slope, length_percentiles, real_ceiling,
                                real_subset_around, target_lengths, unconditional_target_table, usual_length_range)

real_lengths = df["sequence"].str.len().to_numpy()
PERCENTILES = length_percentiles(real_lengths)
TARGETS = target_lengths(real_lengths)  # p10, p30, p50, p70, p90 redondeados
LO_HAB, HI_HAB = usual_length_range(real_lengths)
print(f"Longitud real: n = {real_lengths.size:,}, media {real_lengths.mean():.1f}, mínimo {real_lengths.min()}, máximo {real_lengths.max()}.")
print(f"Rango habitual (percentiles 10-90): {LO_HAB} a {HI_HAB} aa. Tolerancia de acierto: +-{HIT_TOLERANCE} aa.")
ceiling = real_ceiling(real_lengths, TARGETS).set_index("target")["hit_rate"]
tabla_obj = pd.DataFrame({
    "percentil": [f"p{p}" for p in TARGET_PERCENTILES],
    "valor exacto": [PERCENTILES[p] for p in TARGET_PERCENTILES],
    "objetivo t (aa)": TARGETS,
    f"reales con |L-t| <= {HIT_TOLERANCE}": [len(real_subset_around(df["sequence"], t)) for t in TARGETS],
    "techo real (tasa de acierto)": [ceiling[t] for t in TARGETS],
}).set_index("percentil")
display(tabla_obj.round(3))

### b.2 Entrenamiento

Se usa `ConditionalGANConfig.stabilized_constant_noise(conditions=("length",))` (la configuración de la referencia más la condición) y `run_conditional_experiment`, que sigue el mismo protocolo que `run_gan_experiment`: 200 épocas, instantáneas en la 1 y cada 20, 2000 secuencias por instantánea con ruido fijo (semilla 42) y modos `argmax` y `sample`; en cada instantánea se hacen las dos evaluaciones, (i) y (ii). `MAX_PASOS` permanece en `None` (una pasada completa por los datos en cada época); solo se cambia para comprobaciones rápidas del notebook, nunca en la ejecución que cuenta. La primera época incluye la construcción (y, con TensorFlow, la compilación) del modelo; el tiempo de entrenamiento no incluye el de las evaluaciones.

In [ ]:
from amp_gan.conditional_gan import ConditionalGANConfig, run_conditional_experiment

MAX_PASOS = None  # None: una pasada completa por los datos en cada época (solo se cambia para pruebas rápidas)
COND_NAME = "condicional (longitud)"
config_c = ConditionalGANConfig.stabilized_constant_noise(seed=SEED, conditions=("length",))
print(config_c)

res_c = run_conditional_experiment(X, df, config_c, epochs=EPOCHS, snapshot_every=SNAPSHOT_EVERY, n_eval=N_EVAL,
                                   eval_seed=SEED, targets=TARGETS, tol=HIT_TOLERANCE, max_steps_per_epoch=MAX_PASOS)
t_c = res_c.timing
spec_c = res_c.history.spec
print(f"Condición: {spec_c.names}, normalización lineal con extremos {spec_c.low} y {spec_c.high}.")
print(f"Tiempo total: {t_c['total_s']:.1f} s (entrenamiento {t_c['train_s']:.1f} s + evaluaciones {t_c['eval_s']:.1f} s)")
print(f"Por época: media {t_c['mean_epoch_s']:.2f} s, mediana {t_c['median_epoch_s']:.2f} s "
      f"(backend {env['keras_backend']}, dispositivo {decision['device']})")

### b.3 Curvas de pérdida

Las mismas tres medidas por época que en el notebook 02 (pérdida del discriminador con lotes reales y con lotes falsos, y pérdida del generador), de la condicional (continua) y de la referencia sin condicionar (discontinua). Con la etiqueta real suavizada a 0,9 la pérdida del discriminador con lotes reales no puede bajar de ~0,33.

In [ ]:
h_c = res_c.history
epochs_axis_c = np.arange(1, EPOCHS + 1)
panels_c = [("d_loss_real", "Discriminador, lote real"), ("d_loss_fake", "Discriminador, lote falso"), ("g_loss", "Generador")]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, (key, title) in zip(axes, panels_c):
    ax.plot(epochs_axis_c, h_c[key], color="tab:orange", label=COND_NAME)
    ax.plot(ref_hist[REF_NAME]["epoch"], ref_hist[REF_NAME][key], color=COLORS[REF_NAME], ls="--", lw=1, label=REF_NAME)
    ax.set_title(title)
    ax.set_xlabel("Época")
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

hist_c = pd.DataFrame({k: h_c[k] for k in ("d_loss_real", "d_loss_fake", "d_loss", "g_loss")}, index=epochs_axis_c)
display(pd.DataFrame({"primeras 5 épocas": hist_c.head(5).mean(), "épocas 96-100": hist_c.loc[96:100].mean(),
                      "últimas 5 épocas": hist_c.tail(5).mean()}).round(4))
print("¿Algún valor no finito en el historial?", bool((~np.isfinite(hist_c.to_numpy(dtype=float))).any()))

### b.4 Métricas finales de la evaluación (i) (época 200)

Mismo formato y mismas columnas que la tabla a.5, con una columna más: la **condicional por longitud**, evaluada con condiciones tomadas de longitudes reales. La columna de la **referencia** (estabilizada con ruido constante) es la comparación del criterio B; la WGAN-GP se muestra solo como información. Suelos: `real_subset` (lo mínimo alcanzable por el ruido del muestreo) y `random_uniform` (ningún aprendizaje).

In [ ]:
METRICS_B = {**ALL_METRICS, COND_NAME: res_c.metrics}
SEQS_B = {**ALL_SEQS, COND_NAME: {mode: res_c.sequences[mode] for mode in MODES}}
COLORS_B = {**COLORS, COND_NAME: "tab:orange"}

def final_table_b(mode):
    table = pd.DataFrame({name: pick(m, mode) for name, m in METRICS_B.items()}
                         | {"suelo real_subset": pd.Series(floors["real_subset"]),
                            "suelo random_uniform": pd.Series(floors["random_uniform"])})
    return table.loc[cols]

print("Modo sample (principal), época final:")
display(final_table_b("sample").round(3))
print("Modo argmax (secundario), época final:")
display(final_table_b("argmax").round(3))

### b.5 Evaluación (ii): ¿obtiene la longitud que se le pide?

Para cada longitud objetivo `t` (percentiles 10, 30, 50, 70 y 90) se piden 2000 secuencias con esa condición (el mismo `z` para todos los objetivos) y se mide la longitud obtenida. La tabla muestra la época 200 en modo sample (principal): longitud media generada, error absoluto medio, tasa de acierto (±2 aa), fracción de válidas y únicas, tamaño del subconjunto real de referencia (longitud en `[t - 2, t + 2]`) y W1 de carga neta e hidrofobicidad frente a ese subconjunto. Debajo, la **referencia sin condición** (la muestra de `gan_stabc`: no se puede dirigir, así que tiene la misma longitud media para todo `t`) y el **techo real**.

In [ ]:
TCOLS = ["length_mean", "length_mae", "hit_rate", "valid_fraction", "unique_fraction", "n_reference",
         "net_charge_wasserstein", "hydrophobicity_wasserstein"]

def targets_at(mode, epoch=EPOCHS):
    sel = res_c.targets[(res_c.targets["epoch"] == epoch) & (res_c.targets["mode"] == mode)]
    return sel.set_index("target").loc[list(TARGETS)]

ref_tab = unconditional_target_table(ref_seqs[REF_NAME]["sample"], TARGETS).set_index("target")
small = [t for t in TARGETS if res_c.targets.loc[res_c.targets["target"] == t, "n_reference"].iloc[0] < 30]
print(f"Objetivos con menos de 30 secuencias reales de referencia (W1 = NaN): {small if small else 'ninguno'}.\n")

fin_s = targets_at("sample")
tab_s = fin_s[TCOLS].copy()
tab_s["longitud media referencia"] = ref_tab["length_mean"]
tab_s["tasa de acierto referencia"] = ref_tab["hit_rate"]
tab_s["techo real"] = ceiling
print(f"Evaluación por objetivos, modo sample, época {EPOCHS} (n = {N_EVAL} por objetivo):")
display(tab_s.round(3))
print(f"Referencia sin condición (gan_stabc, {len(ref_seqs[REF_NAME]['sample'])} secuencias): longitud media {ref_tab['length_mean'].iloc[0]:.2f}; "
      f"la condicional pide t = {list(TARGETS)}.")

fin_a = targets_at("argmax")
tab_a = fin_a[TCOLS].copy()
tab_a["tasa de acierto referencia"] = ref_tab["hit_rate"]
tab_a["techo real"] = ceiling
print(f"\nModo argmax (secundario), época {EPOCHS}:")
display(tab_a.round(3))

**Longitud generada frente a longitud pedida** (época final). Cada punto es la longitud media de las 2000 secuencias pedidas con ese objetivo y la barra, su desviación típica. La línea discontinua gris es la **diagonal ideal** (longitud generada = longitud pedida), la línea naranja la regresión lineal de los cinco puntos y la horizontal azul la longitud media de la referencia sin condición, que no depende de lo que se pida.

In [ ]:
def target_lengths_gen(mode, t):
    return np.array([len(s) for s in res_c.target_sequences[mode][t]], dtype=float)

tt = np.array(TARGETS, dtype=float)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharex=True, sharey=True)
for ax, mode in zip(axes, ("sample", "argmax")):
    means = np.array([target_lengths_gen(mode, t).mean() for t in TARGETS])
    stds = np.array([target_lengths_gen(mode, t).std(ddof=1) for t in TARGETS])
    slope_m, icpt_m = length_control_slope(TARGETS, means)
    lim = [0, max(tt.max(), (means + stds).max()) + 3]
    ax.plot(lim, lim, color="gray", ls="--", label="diagonal ideal (generada = pedida)")
    ax.axhline(ref_tab["length_mean"].iloc[0], color=COLORS[REF_NAME], ls="-.", label=f"referencia sin condición (media {ref_tab['length_mean'].iloc[0]:.1f})")
    ax.plot(lim, [slope_m * v + icpt_m for v in lim], color="tab:orange", alpha=0.5, label=f"regresión: pendiente {slope_m:.2f}")
    ax.errorbar(tt, means, yerr=stds, fmt="o", color="tab:orange", capsize=3, label=f"{COND_NAME} (media +- desv. típica)")
    ax.set_title(f"Longitud generada frente a pedida - {mode} (época {EPOCHS})")
    ax.set_xlabel("Longitud pedida (aa)")
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.legend(fontsize=8, loc="upper left")
axes[0].set_ylabel("Longitud generada (aa)")
fig.tight_layout()
plt.show()

**Tasa de acierto (±2 aa) por objetivo.** Barras: techo real (fracción de datos reales dentro de la tolerancia, lo que daría una muestra con la distribución real sin dirigir), la referencia sin condición (`gan_stabc`) y la condicional (sample y argmax).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
xs = np.arange(len(TARGETS))
w_bar = 0.2
series_hr = [("techo real", ceiling.loc[list(TARGETS)].to_numpy(), "gray", None),
             (REF_NAME, ref_tab["hit_rate"].loc[list(TARGETS)].to_numpy(), COLORS[REF_NAME], None),
             (f"{COND_NAME}, sample", fin_s["hit_rate"].to_numpy(), "tab:orange", None),
             (f"{COND_NAME}, argmax", fin_a["hit_rate"].to_numpy(), "tab:orange", "//")]
for i, (label, vals, color, hatch) in enumerate(series_hr):
    bars = ax.bar(xs + (i - 1.5) * w_bar, vals, width=w_bar, color=color, hatch=hatch, alpha=0.9 if hatch is None else 0.5, label=label)
    for rect, v in zip(bars, vals):
        ax.text(rect.get_x() + rect.get_width() / 2, v, f"{v:.2f}", ha="center", va="bottom", fontsize=7)
ax.set_xticks(xs)
ax.set_xticklabels([f"t = {t}\n(p{p})" for t, p in zip(TARGETS, TARGET_PERCENTILES)])
ax.set_ylabel(f"Tasa de acierto (|L - t| <= {HIT_TOLERANCE})")
ax.set_title(f"Tasa de acierto por longitud pedida (época {EPOCHS})")
ax.set_ylim(0, 1.12)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
plt.show()

### b.6 Evolución por instantánea

**Evaluación (i).** Las mismas métricas que en a.8 para la referencia, la WGAN-GP y la condicional (continua: sample; discontinua: argmax). La línea gris punteada es el suelo `real_subset` (en la longitud media, la media real).

In [ ]:
SHOWN_B = [REF_NAME, "WGAN-GP", COND_NAME]
fig, axes = plt.subplots(2, 3, figsize=(14, 7.6))
for ax, (col, title, real_value) in zip(axes.ravel(), panels):
    for name in SHOWN_B:
        for mode, ls in (("sample", "-"), ("argmax", "--")):
            sel = METRICS_B[name][METRICS_B[name]["mode"] == mode]
            ax.plot(sel["epoch"], sel[col], ls, marker="o", ms=3.5, color=COLORS_B[name])
    ax.axhline(real_value if real_value is not None else floors["real_subset"][col], color="gray", ls=":")
    ax.set_title(title)
    ax.set_xlabel("Época")
    ax.set_ylabel(title)
handles = [Line2D([], [], color=COLORS_B[n_], ls=ls_, marker="o", ms=3.5, label=f"{n_}, {m_}")
           for n_ in SHOWN_B for m_, ls_ in (("sample", "-"), ("argmax", "--"))]
handles.append(Line2D([], [], color="gray", ls=":", label="suelo real_subset (media real en la longitud media)"))
fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=8, frameon=False)
fig.tight_layout(rect=(0, 0.1, 1, 1))
plt.show()

snap_b = pd.concat({
    "W1 longitud": pd.DataFrame({n: snap_series(METRICS_B[n], "length_wasserstein") for n in SHOWN_B}),
    f"longitud media (real = {real_mean_len:.2f})": pd.DataFrame({n: snap_series(METRICS_B[n], "length_mean") for n in SHOWN_B}),
}, axis=1)
snap_b.index.name = "época"
print(f"W1 de longitud y longitud media por instantánea (modo sample); suelo real_subset de la W1 de longitud: {floors['real_subset']['length_wasserstein']:.3f}")
display(snap_b.round(3))

**Evaluación (ii).** Por instantánea y objetivo (modo sample): longitud media generada (con la línea del objetivo), error absoluto medio y tasa de acierto. Debajo, la pendiente de la regresión de la longitud media generada sobre la pedida en cada instantánea (1 = control perfecto, 0 = la longitud no depende de la condición).

In [ ]:
tcs = res_c.targets[res_c.targets["mode"] == "sample"]
cmap_t = plt.get_cmap("viridis")
fig, axes = plt.subplots(1, 3, figsize=(15, 3.9))
for i, t in enumerate(TARGETS):
    s_t = tcs[tcs["target"] == t]
    color = cmap_t(i / max(len(TARGETS) - 1, 1))
    axes[0].plot(s_t["epoch"], s_t["length_mean"], marker="o", ms=3.5, color=color, label=f"t = {t}")
    axes[0].axhline(t, color=color, ls=":", lw=1)
    axes[1].plot(s_t["epoch"], s_t["length_mae"], marker="o", ms=3.5, color=color, label=f"t = {t}")
    axes[2].plot(s_t["epoch"], s_t["hit_rate"], marker="o", ms=3.5, color=color, label=f"t = {t}")
for ax, title in zip(axes, ("Longitud media generada (aa; punteado: objetivo)", "Error absoluto medio |L - t| (aa)",
                            f"Tasa de acierto (|L - t| <= {HIT_TOLERANCE})")):
    ax.set_title(title)
    ax.set_xlabel("Época")
axes[2].legend(fontsize=8, ncol=2)
fig.tight_layout()
plt.show()

pend = pd.DataFrame({
    mode: {e: length_control_slope(TARGETS, g.set_index("target").loc[list(TARGETS), "length_mean"])[0]
           for e, g in res_c.targets[res_c.targets["mode"] == mode].groupby("epoch")}
    for mode in ("sample", "argmax")})
pend.index.name = "época"
print("Pendiente de la longitud media generada frente a la pedida, por instantánea:")
display(pend.T.round(3))

### b.7 Distribuciones

**Evaluación (i), época final.** Histogramas normalizados de longitud, carga neta e hidrofobicidad: real (gris), referencia (azul), WGAN-GP (morado) y condicional (naranja), con los mismos intervalos y ejes en cada panel. Arriba, modo sample; abajo, modo argmax. Solo se usan las secuencias generadas válidas.

In [ ]:
valid_b = {name: {mode: [s for s in SEQS_B[name][mode] if is_valid(s)] for mode in MODES} for name in SEQS_B}
prof_b = {name: {mode: profile_sequences(v) if v else None for mode, v in d.items()} for name, d in valid_b.items()}
SHOWN_D = [REF_NAME, "WGAN-GP", COND_NAME]

edges_b = {}
for prop, _ in PROPS:
    allv = np.concatenate([prof_real.values(prop)] + [prof_b[n][m].values(prop) for n in SHOWN_D for m in MODES if prof_b[n][m] is not None])
    lo, hi = np.floor(allv.min()), np.ceil(allv.max())
    edges_b[prop] = np.arange(lo - 0.5, hi + 1.5) if prop == "length" else np.linspace(lo, hi, 41)

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex="col", sharey="col")
for r, mode in enumerate(("sample", "argmax")):
    for c, (prop, xlabel) in enumerate(PROPS):
        ax = axes[r, c]
        real = prof_real.values(prop)
        ax.hist(real, bins=edges_b[prop], weights=np.full(real.size, 1 / real.size), color="gray", alpha=0.4, label=f"real (n={real.size:,})")
        for name in SHOWN_D:
            p = prof_b[name][mode]
            if p is not None:
                g = p.values(prop)
                ax.hist(g, bins=edges_b[prop], weights=np.full(g.size, 1 / g.size), histtype="step", lw=1.6, color=COLORS_B[name],
                        label=f"{name} (n={g.size:,})")
        ax.set_title(f"{xlabel} - {mode}")
        ax.set_xlabel(xlabel)
        if c == 0:
            ax.set_ylabel("Fracción de secuencias")
        ax.legend(fontsize=7, loc="upper right")
for c in range(3):
    axes[0, c].set_ylim(0, max(axes[r, c].get_ylim()[1] for r in range(2)) * 1.3)
fig.tight_layout()
plt.show()

**Evaluación (ii): distribución de la longitud generada para cada longitud pedida** (modo sample, época final). La línea vertical marca el objetivo y la banda, la tolerancia de acierto (±2 aa); en gris, la longitud real (todas las secuencias).

In [ ]:
fig, axes = plt.subplots(1, len(TARGETS), figsize=(3.4 * len(TARGETS), 3.6), sharey=True)
bins_len = np.arange(-0.5, 61.5)
for ax, (t, color) in zip(np.atleast_1d(axes), zip(TARGETS, [cmap_t(i / max(len(TARGETS) - 1, 1)) for i in range(len(TARGETS))])):
    g = target_lengths_gen("sample", t)
    ax.hist(real_lengths, bins=bins_len, weights=np.full(real_lengths.size, 1 / real_lengths.size), color="gray", alpha=0.35, label="real")
    ax.hist(g, bins=bins_len, weights=np.full(g.size, 1 / g.size), histtype="stepfilled", color=color, alpha=0.7, label=f"pedida t = {t}")
    ax.axvspan(t - HIT_TOLERANCE, t + HIT_TOLERANCE, color="k", alpha=0.08)
    ax.axvline(t, color="k", lw=1)
    ax.set_title(f"t = {t}: acierto {fin_s.loc[t, 'hit_rate']:.2f}")
    ax.set_xlabel("Longitud generada (aa)")
    ax.legend(fontsize=7)
np.atleast_1d(axes)[0].set_ylabel("Fracción de secuencias")
fig.tight_layout()
plt.show()

### b.8 Composición de aminoácidos

Fracción media de cada aminoácido en las secuencias reales y en las generadas válidas de la referencia y de la condicional (evaluación (i)), ordenados por abundancia en las reales, con el mismo eje vertical en los dos modos.

In [ ]:
SHOWN_C = [REF_NAME, COND_NAME]
comp_b = pd.DataFrame({"real": prof_real.composition}, index=list(STANDARD_ALPHABET))
for mode in MODES:
    for name in SHOWN_C:
        comp_b[f"{name}|{mode}"] = prof_b[name][mode].composition if prof_b[name][mode] is not None else np.nan
comp_b = comp_b.sort_values("real", ascending=False)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, sharey=True)
x_aa = np.arange(len(comp_b))
w_aa = 0.27
for ax, mode in zip(axes, ("sample", "argmax")):
    series = [("real", "real", "gray")] + [(f"{n}|{mode}", n, COLORS_B[n]) for n in SHOWN_C]
    for i, (col, label, color) in enumerate(series):
        ax.bar(x_aa + (i - 1) * w_aa, comp_b[col] * 100, width=w_aa, color=color, label=label)
    ax.set_ylabel("Composición media (%)")
    ax.set_title(f"Composición media de aminoácidos - {mode} (época final)")
    ax.legend()
axes[-1].set_xticks(x_aa)
axes[-1].set_xticklabels(comp_b.index)
axes[-1].set_xlabel("Aminoácido (ordenados por abundancia en las secuencias reales)")
fig.tight_layout()
plt.show()

### b.9 Secuencias de ejemplo

Cinco secuencias pedidas con longitud igual a los percentiles 10, 50 y 90 (modo sample, época final), con su longitud, carga neta e hidrofobicidad. Las no válidas aparecen sin propiedades. Debajo, el recuento de repetición de la evaluación (i) (secuencias válidas, distintas entre ellas y veces que aparece la más repetida).

In [ ]:
for p in (10, 50, 90):
    t = TARGETS[TARGET_PERCENTILES.index(p)]
    print(f"Longitud pedida t = {t} aa (percentil {p}), modo sample:")
    ex = example_table(res_c.target_sequences["sample"][t], k=5)
    ex.insert(0, "pedida", t)
    display(ex)

rep_c = []
for name in (REF_NAME, COND_NAME):
    for mode in ("sample", "argmax"):
        valid = [s for s in SEQS_B[name][mode] if is_valid(s)]
        cnt = Counter(valid)
        top_seq, top_n = cnt.most_common(1)[0] if cnt else ("", 0)
        rep_c.append({"variante": name, "modo": mode, "generadas": len(SEQS_B[name][mode]), "válidas": len(valid),
                      "distintas": len(cnt), "más repetida (veces)": top_n, "secuencia más repetida": top_seq[:40]})
print("Recuentos de repetición, evaluación (i) (época final):")
display(pd.DataFrame(rep_c).set_index(["variante", "modo"]))

### b.10 Comparación con la ejecución del script

El mismo experimento se ejecutó con `python src/amp_gan/run_conditional_gan.py --variant cgan_length --auto-backend` y sus resultados están en `context/data/processed/cgan_length_*`. Si ese archivo existe, se compara con el de esta ejecución (época final, modo sample, evaluación (i)); las diferencias, si las hay, proceden del backend o del dispositivo, no de la configuración (misma semilla y mismos parámetros).

In [ ]:
f_script = PROCESSED / "cgan_length_metrics.csv"
if f_script.exists():
    m_script = pd.read_csv(f_script)
    m_script = m_script[m_script["epoch"].notna()].astype({"epoch": int})
    info_script = json.loads((PROCESSED / "cgan_length_run_info.json").read_text(encoding="utf-8"))
    print(f"Script: backend {info_script['keras_backend']} en {info_script['device_used']}; esta ejecución: {env['keras_backend']} en {decision['device']}.")
    comp_script = pd.DataFrame({"script": pick(m_script, "sample"), "esta ejecución": pick(res_c.metrics, "sample")}).loc[cols]
    display(comp_script.round(3))
else:
    print("No existe cgan_length_metrics.csv (genéralo con python src/amp_gan/run_conditional_gan.py --variant cgan_length).")

### b.11 Conclusiones (descriptivas)

Primero se evalúan los criterios A y B fijados de antemano (modo sample, época 200, sin elegir la mejor época), condición por condición y con las cifras de esta ejecución. Después se describen, sin interpretar causas, las demás métricas, la evolución, los tiempos y el comportamiento del entrenamiento. El texto se genera con el código de la celda siguiente.

In [ ]:
UNIQUE_MIN, SLOPE_RANGE, MAE_MAX = 0.95, (0.8, 1.2), 3.0
SI_NO = lambda ok: "SE CUMPLE" if ok else "NO se cumple"

print("CRITERIO A - control de la longitud (modo sample, época 200)")
slope_c, icpt_c = length_control_slope(TARGETS, fin_s["length_mean"])
a1 = SLOPE_RANGE[0] <= slope_c <= SLOPE_RANGE[1]
print(f"  1. Pendiente de la longitud media generada frente a la pedida: {slope_c:.3f} (ordenada {icpt_c:.2f}) -> {SI_NO(a1)} (se pedía entre {SLOPE_RANGE[0]} y {SLOPE_RANGE[1]}).")
mae_c = float(fin_s["length_mae"].mean())
a2 = mae_c <= MAE_MAX
print(f"  2. Error absoluto medio |L - t| promediado sobre los objetivos: {mae_c:.2f} aa -> {SI_NO(a2)} (se pedía <= {MAE_MAX:g} aa).")
a3 = []
for t in TARGETS:
    ok = bool(fin_s.loc[t, "hit_rate"] > ref_tab.loc[t, "hit_rate"])
    a3.append(ok)
    print(f"     t = {t:2d}: tasa de acierto condicional {fin_s.loc[t, 'hit_rate']:.3f} frente a referencia {ref_tab.loc[t, 'hit_rate']:.3f} "
          f"(techo real {ceiling[t]:.3f}) -> {'supera' if ok else 'NO supera'}")
a3_ok = all(a3)
print(f"  3. Tasa de acierto superior a la de la referencia en los {len(TARGETS)} objetivos: la supera en {sum(a3)} de {len(TARGETS)} -> {SI_NO(a3_ok)}.")
A_ok = a1 and a2 and a3_ok
print(f"  Resultado A: {sum([a1, a2, a3_ok])} de 3 condiciones -> la condicional {'SÍ' if A_ok else 'NO'} controla la longitud según el criterio.\n")

c_s, r_s = pick(res_c.metrics, "sample"), pick(ALL_METRICS[REF_NAME], "sample")
print("CRITERIO B - fidelidad con condiciones reales (evaluación (i), modo sample, época 200)")
b1 = bool(c_s["length_wasserstein"] <= r_s["length_wasserstein"])
print(f"  1. W1 de longitud: referencia {r_s['length_wasserstein']:.3f}, condicional {c_s['length_wasserstein']:.3f} -> {SI_NO(b1)} (se pedía <= la de la referencia).")
b2 = bool(c_s["unique_fraction"] >= UNIQUE_MIN)
print(f"  2. unique_fraction: referencia {r_s['unique_fraction']:.3f}, condicional {c_s['unique_fraction']:.3f} -> {SI_NO(b2)} (se pedía >= {UNIQUE_MIN}).")
B_ok = b1 and b2
print(f"  Resultado B: {sum([b1, b2])} de 2 condiciones -> la condicional {'NO empeora' if B_ok else 'SÍ empeora (según el criterio)'} a la referencia.")
print("  Informativas, sin condición de aprobado (referencia entre paréntesis; WGAN-GP entre corchetes):")
w_s = pick(ALL_METRICS["WGAN-GP"], "sample")
for col, label in (("net_charge_wasserstein", "W1 de carga neta"), ("hydrophobicity_wasserstein", "W1 de hidrofobicidad"),
                   ("composition_tv", "variación total de composición")):
    print(f"     {label}: {c_s[col]:.3f} ({r_s[col]:.3f}) [{w_s[col]:.3f}]")
print("\nEs una sola ejecución con la semilla 42: no se estima la variabilidad entre semillas ni se ajusta ningún hiperparámetro.")

print()
print(f"Rango habitual p10-p90: {LO_HAB}-{HI_HAB} aa; objetivos {list(TARGETS)}; tolerancia +-{HIT_TOLERANCE} aa.")
print(f"[sample] época {EPOCHS}, evaluación (i): condicional (referencia; suelos real_subset / random_uniform)")
fl, ru = floors["real_subset"], floors["random_uniform"]
for col, label in (("valid_fraction", "válidas"), ("unique_fraction", "únicas"), ("novel_fraction", "nuevas"),
                   ("length_mean", "longitud media"), ("length_wasserstein", "W1 longitud"),
                   ("net_charge_mean", "carga media"), ("net_charge_wasserstein", "W1 carga"),
                   ("hydrophobicity_mean", "GRAVY medio"), ("hydrophobicity_wasserstein", "W1 hidrofobicidad"),
                   ("composition_tv", "variación total de composición")):
    suelos = f"{fl[col]:.3f} / {ru[col]:.3f}" if col in fl else "-"
    print(f"  - {label}: {c_s[col]:.3f} ({r_s[col]:.3f}); suelos {suelos}")
c_a, r_a = pick(res_c.metrics, "argmax"), pick(ALL_METRICS[REF_NAME], "argmax")
print(f"[argmax] época {EPOCHS}: unique_fraction {c_a['unique_fraction']:.3f} ({r_a['unique_fraction']:.3f}); "
      f"W1 longitud {c_a['length_wasserstein']:.3f} ({r_a['length_wasserstein']:.3f}); valid_fraction {c_a['valid_fraction']:.3f} ({r_a['valid_fraction']:.3f}).")

print(f"\nEvaluación por objetivos (sample, época {EPOCHS}): longitud media generada " +
      "; ".join(f"t={t}: {fin_s.loc[t, 'length_mean']:.1f}" for t in TARGETS) + ".")
print("  unique_fraction por objetivo: " + "; ".join(f"t={t}: {fin_s.loc[t, 'unique_fraction']:.3f}" for t in TARGETS)
      + ". valid_fraction: " + "; ".join(f"t={t}: {fin_s.loc[t, 'valid_fraction']:.3f}" for t in TARGETS) + ".")
print(f"  Evaluación por objetivos (argmax): pendiente {length_control_slope(TARGETS, fin_a['length_mean'])[0]:.3f}, MAE medio {fin_a['length_mae'].mean():.2f} aa, "
      f"tasas de acierto " + "; ".join(f"t={t}: {fin_a.loc[t, 'hit_rate']:.3f}" for t in TARGETS) + ".")
print(f"  W1 de carga neta frente al subconjunto real (sample): " + "; ".join(f"t={t}: {fin_s.loc[t, 'net_charge_wasserstein']:.2f}" for t in TARGETS)
      + ". W1 de hidrofobicidad: " + "; ".join(f"t={t}: {fin_s.loc[t, 'hydrophobicity_wasserstein']:.3f}" for t in TARGETS) + ".")

print()
sl = pend["sample"]
print("Pendiente por instantánea (sample): " + "; ".join(f"{e}: {v:.2f}" for e, v in sl.items()) + ".")
w1c = snap_series(res_c.metrics, "length_wasserstein")
w1r = snap_series(ALL_METRICS[REF_NAME], "length_wasserstein")
print("W1 de longitud por instantánea (sample), condicional: " + "; ".join(f"{e}: {v:.2f}" for e, v in w1c.items()) + ".")
print(f"Entre las épocas 20 y {EPOCHS}, la W1 de longitud de la condicional va de {w1c[w1c.index >= 20].max():.2f} (máximo) a {w1c[w1c.index >= 20].min():.2f} (mínimo); "
      f"la de la referencia, de {w1r[w1r.index >= 20].max():.2f} a {w1r[w1r.index >= 20].min():.2f}.")

f5c, l5c = hist_c.head(5).mean(), hist_c.tail(5).mean()
print(f"\nPérdidas medias (primeras 5 -> últimas 5 épocas): discriminador real {f5c['d_loss_real']:.3f} -> {l5c['d_loss_real']:.3f}; "
      f"discriminador falso {f5c['d_loss_fake']:.3f} -> {l5c['d_loss_fake']:.3f}; generador {f5c['g_loss']:.3f} -> {l5c['g_loss']:.3f}.")
hr = ref_hist[REF_NAME].tail(5).mean()
print(f"Referencia (últimas 5 épocas): discriminador real {hr['d_loss_real']:.3f}; falso {hr['d_loss_fake']:.3f}; generador {hr['g_loss']:.3f}.")
print("Valores no finitos en el historial:", int((~np.isfinite(hist_c.to_numpy(dtype=float))).sum()),
      "; NaN en las métricas (i):", int(res_c.metrics.drop(columns=["epoch", "mode"]).isna().sum().sum()),
      "; NaN en la evaluación por objetivos (aparte de las W1 con subconjunto pequeño):",
      int(res_c.targets.drop(columns=["epoch", "mode", "net_charge_wasserstein", "hydrophobicity_wasserstein"]).isna().sum().sum()), ".")
rcc = pd.DataFrame(rep_c)
for name in (COND_NAME, REF_NAME):
    sub = rcc[rcc["variante"] == name]
    print(f"{name}: " + "; ".join(f"[{d['modo']}] {d['distintas']} secuencias distintas de {d['válidas']} válidas "
                                    f"(la más repetida, {d['más repetida (veces)']} veces)" for _, d in sub.iterrows()) + ".")
print(f"Tiempo: condicional {t_c['mean_epoch_s']:.2f} s/época de media ({t_c['total_s']:.0f} s en total, backend {env['keras_backend']} "
      f"en {decision['device']}); referencia {ref_info[REF_NAME]['timing_s']['mean_epoch_s']:.2f} s/época "
      f"({ref_info[REF_NAME]['keras_backend']} en {ref_info[REF_NAME]['device_used']}).")

### b.12 Nota sobre tiempos

La condicional tiene un coste por época parecido al de la vanilla estabilizada con ruido constante, algo mayor (con torch en GPU, 2,56 s/época frente a 2,36 s de la referencia, ~9 % más). A las evaluaciones por instantánea se suman las 5 tandas de 2000 secuencias por objetivo y modo, que siguen siendo una fracción pequeña del total (~8 s con torch en GPU y ~11 s con TensorFlow en CPU en las ejecuciones del script). Los tiempos medidos están en la nota de la introducción; la celda anterior imprime los de esta ejecución.